# Genetic Algorithm Feature Selection

This notebook applies a Genetic Algorithm (GA) to the filtered Mordred descriptor set to identify a high-performing subset of features for dual JNK3/GSK3β classification. Feature subsets are evaluated using XGBoost and validation MCC, with the test set excluded from GA feature selection. The selected GA descriptors are then compared with the full Mordred and mRMR feature sets and finally evaluated on the held-out scaffold test set.

In [ ]:
# Import pandas because the saved datasets are stored as pandas files
import pandas as pd

# Load the 626 filtered Mordred descriptors for the training set
X_train = pd.read_pickle("mordred_training_features_filtered.pkl")

# Load the training labels
y_train = pd.read_pickle("mordred_training_labels.pkl")

# Load the same filtered Mordred descriptors for the validation set
X_valid = pd.read_pickle("mordred_validation_features_filtered.pkl")

# Load the validation labels
y_valid = pd.read_pickle("mordred_validation_labels.pkl")

# Load the file containing the names of the 626 filtered descriptors
descriptor_names_df = pd.read_csv(
    "mordred_filtered_descriptor_names.csv"
)

# The labels may have been saved as a one-column DataFrame
# This converts them into a simple one-dimensional Series if needed
if isinstance(y_train, pd.DataFrame):
    y_train = y_train.squeeze("columns")

if isinstance(y_valid, pd.DataFrame):
    y_valid = y_valid.squeeze("columns")

# Remove any extra index column that may have been saved in the CSV file
descriptor_names_df = descriptor_names_df.loc[
    :,
    ~descriptor_names_df.columns.str.startswith("Unnamed")
]

# Use the first remaining column as the descriptor-name list
descriptor_names = (
    descriptor_names_df.iloc[:, 0]
    .astype(str)
    .tolist()
)

# Display the shape of each dataset
print("Training features shape:", X_train.shape)
print("Training labels shape:", y_train.shape)

print("\nValidation features shape:", X_valid.shape)
print("Validation labels shape:", y_valid.shape)

print("\nNumber of saved descriptor names:", len(descriptor_names))

# Check whether the training and validation feature columns are identical
same_feature_order = list(X_train.columns) == list(X_valid.columns)

print(
    "Training and validation descriptors are in the same order:",
    same_feature_order
)

# Check whether the saved descriptor names match the feature columns
descriptor_names_match = (
    list(X_train.columns.astype(str)) == descriptor_names
)

print(
    "Saved descriptor names match the training columns:",
    descriptor_names_match
)

# Check that every molecule has one matching classification label
train_rows_match = len(X_train) == len(y_train)
valid_rows_match = len(X_valid) == len(y_valid)

print("\nTraining rows and labels match:", train_rows_match)
print("Validation rows and labels match:", valid_rows_match)

# Confirm that the GA is beginning with exactly 626 descriptors
print(
    "Exactly 626 descriptors are available:",
    X_train.shape[1] == 626 and X_valid.shape[1] == 626
)

# Display the class distribution without changing the data
print("\nTraining label counts:")
print(y_train.value_counts().sort_index())

print("\nValidation label counts:")
print(y_valid.value_counts().sort_index())

# Confirm that no test-set file has been loaded in this notebook
print("\nThe test set has not been loaded or used.")

In [ ]:
import numpy as np

# Use smaller data types to reduce memory usage
X_train = X_train.astype(np.float32, copy=False)
X_valid = X_valid.astype(np.float32, copy=False)

y_train = y_train.astype(np.int8, copy=False)
y_valid = y_valid.astype(np.int8, copy=False)

# Check for missing or infinite values
print("Missing values in training set:", X_train.isna().sum().sum())
print("Missing values in validation set:", X_valid.isna().sum().sum())

print(
    "Infinite values in training set:",
    np.isinf(X_train.to_numpy()).sum()
)

print(
    "Infinite values in validation set:",
    np.isinf(X_valid.to_numpy()).sum()
)

# Check the final data types and memory usage
print("\nTraining feature data type:", X_train.dtypes.unique())
print("Validation feature data type:", X_valid.dtypes.unique())

print(
    "Training data memory:",
    round(X_train.memory_usage(deep=True).sum() / 1024**2, 2),
    "MB"
)

print(
    "Validation data memory:",
    round(X_valid.memory_usage(deep=True).sum() / 1024**2, 2),
    "MB"
)

In [ ]:
import numpy as np
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import matthews_corrcoef

# Use balanced weights because there are more dual candidates in the dataset
train_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_train
)

# Keep the same XGBoost settings used for the mRMR models
xgb_params = {
    "n_estimators": 1000,
    "learning_rate": 0.05,
    "max_depth": 6,
    "min_child_weight": 3,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "tree_method": "hist",
    "device": "cuda",
    "random_state": 42,
    "early_stopping_rounds": 50
}

# Train the full 626-descriptor baseline before starting the GA
baseline_model = XGBClassifier(**xgb_params)

baseline_model.fit(
    X_train,
    y_train,
    sample_weight=train_weights,
    eval_set=[(X_valid, y_valid)],
    verbose=False
)

# Calculate validation predictions using the default threshold of 0.50
baseline_probabilities = baseline_model.predict_proba(X_valid)[:, 1]
baseline_predictions = (baseline_probabilities >= 0.50).astype(int)

baseline_mcc = matthews_corrcoef(
    y_valid,
    baseline_predictions
)

print("Best boosting round:", baseline_model.best_iteration)
print("Validation MCC:", round(baseline_mcc, 3))

## Full Mordred baseline check

The XGBoost model using all 626 Mordred descriptors achieved a validation MCC of 0.635 using GPU training. The best boosting iteration was 982.

This result was lower than the MCC of 0.703 obtained in the earlier Mordred experiment. Therefore, the same model was also checked using CPU training before starting the Genetic Algorithm.

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import matthews_corrcoef

# Train the same model using CPU settings from the earlier Mordred notebook
cpu_baseline_model = XGBClassifier(
    objective="binary:logistic",
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=3,
    subsample=0.80,
    colsample_bytree=0.80,
    reg_alpha=0.10,
    reg_lambda=1.00,
    tree_method="hist",
    eval_metric="logloss",
    early_stopping_rounds=50,
    random_state=42,
    n_jobs=-1
)

cpu_baseline_model.fit(
    X_train,
    y_train,
    sample_weight=train_weights,
    eval_set=[(X_valid, y_valid)],
    verbose=False
)

# Calculate MCC at the default threshold of 0.50
cpu_probabilities = cpu_baseline_model.predict_proba(X_valid)[:, 1]
cpu_predictions = (cpu_probabilities >= 0.50).astype(int)

cpu_baseline_mcc = matthews_corrcoef(
    y_valid,
    cpu_predictions
)

print("Best boosting round:", cpu_baseline_model.best_iteration)
print("CPU validation MCC:", round(cpu_baseline_mcc, 3))

## Selection of the XGBoost training setting

The CPU-based XGBoost model achieved a validation MCC of 0.703, which matched the result obtained in the earlier full Mordred experiment. The GPU version produced a different MCC of 0.635.

Therefore, CPU-based XGBoost was selected for the Genetic Algorithm so that the same training method and hyperparameters were used for the full Mordred, mRMR and GA feature-selection comparisons.

In [ ]:
# Keep one fixed set of XGBoost settings for all GA evaluations
ga_xgb_params = {
    "objective": "binary:logistic",
    "n_estimators": 1000,
    "learning_rate": 0.05,
    "max_depth": 6,
    "min_child_weight": 3,
    "subsample": 0.80,
    "colsample_bytree": 0.80,
    "reg_alpha": 0.10,
    "reg_lambda": 1.00,
    "tree_method": "hist",
    "eval_metric": "logloss",
    "early_stopping_rounds": 50,
    "random_state": 42,
    "n_jobs": -1
}

print("Number of descriptors available to the GA:", X_train.shape[1])
print("GA fitness metric: MCC")
print("XGBoost training device: CPU")
print("Test set used: No")

## Genetic Algorithm settings

The Genetic Algorithm was set to begin with all 626 filtered Mordred descriptors. MCC was selected as the fitness measure, and CPU-based XGBoost was retained to ensure consistency with the earlier Mordred and mRMR experiments. The test set was not used during feature selection.

In [ ]:
# Convert the data to NumPy arrays for faster repeated model training
X_train_array = X_train.to_numpy(copy=False)
X_valid_array = X_valid.to_numpy(copy=False)

y_train_array = y_train.to_numpy()
y_valid_array = y_valid.to_numpy()

# Store previously evaluated feature subsets to avoid repeating the same model
fitness_cache = {}

def evaluate_feature_subset(individual):
    # Convert the binary chromosome into a Boolean feature mask
    feature_mask = np.asarray(individual, dtype=bool)

    # Return a poor score if no descriptor is selected
    if feature_mask.sum() == 0:
        return -1.0

    # Use the feature mask as the cache key
    cache_key = feature_mask.tobytes()

    # Return the saved MCC if this subset was already evaluated
    if cache_key in fitness_cache:
        return fitness_cache[cache_key]

    # Select the descriptors represented by 1 in the chromosome
    X_train_selected = X_train_array[:, feature_mask]
    X_valid_selected = X_valid_array[:, feature_mask]

    # Train XGBoost using the fixed settings
    model = XGBClassifier(**ga_xgb_params)

    model.fit(
        X_train_selected,
        y_train_array,
        sample_weight=train_weights,
        eval_set=[(X_valid_selected, y_valid_array)],
        verbose=False
    )

    # Calculate MCC using the default threshold of 0.50
    validation_probabilities = model.predict_proba(
        X_valid_selected
    )[:, 1]

    validation_predictions = (
        validation_probabilities >= 0.50
    ).astype(int)

    fitness = matthews_corrcoef(
        y_valid_array,
        validation_predictions
    )

    # Save the result so the same subset is not trained again
    fitness_cache[cache_key] = fitness

    return fitness

print("GA fitness function created successfully.")

## Fitness function

A fitness function was created to evaluate each descriptor subset generated by the Genetic Algorithm. Each chromosome contains 626 binary values, where `1` means that a descriptor is selected and `0` means that it is excluded.

For every selected subset, XGBoost is trained using the fixed model settings and its validation MCC is returned as the fitness score. Previously tested subsets are stored in a cache to prevent unnecessary repeated model training.

In [ ]:
import random
from deap import base, creator, tools

# Set the random seed so the GA results can be reproduced
random.seed(42)
np.random.seed(42)

# Create a fitness class that will maximise MCC
if not hasattr(creator, "FitnessMax"):
    creator.create("FitnessMax", base.Fitness, weights=(1.0,))

# Create one GA individual containing 626 binary values
if not hasattr(creator, "Individual"):
    creator.create(
        "Individual",
        list,
        fitness=creator.FitnessMax
    )

toolbox = base.Toolbox()

# Each descriptor is randomly selected or excluded
toolbox.register(
    "attribute",
    random.randint,
    0,
    1
)

# Create one chromosome with 626 descriptor positions
toolbox.register(
    "individual",
    tools.initRepeat,
    creator.Individual,
    toolbox.attribute,
    n=626
)

# Create a population containing several chromosomes
toolbox.register(
    "population",
    tools.initRepeat,
    list,
    toolbox.individual
)

# DEAP requires the fitness value to be returned as a tuple
def ga_fitness(individual):
    return (evaluate_feature_subset(individual),)

toolbox.register("evaluate", ga_fitness)

# Exchange parts of two chromosomes
toolbox.register(
    "mate",
    tools.cxTwoPoint
)

# Give each descriptor a small chance of changing between 0 and 1
toolbox.register(
    "mutate",
    tools.mutFlipBit,
    indpb=1 / 626
)

# Select better-performing individuals for the next generation
toolbox.register(
    "select",
    tools.selTournament,
    tournsize=3
)

# Small pilot settings to check that the GA works correctly
pilot_population_size = 6
pilot_generations = 3
pilot_crossover_probability = 0.80
pilot_mutation_probability = 0.20

print("Chromosome length:", 626)
print("Pilot population size:", pilot_population_size)
print("Pilot generations:", pilot_generations)
print("Crossover probability:", pilot_crossover_probability)
print("Mutation probability:", pilot_mutation_probability)

## Full Genetic Algorithm settings

The full GA was configured with a population of 40 individuals and 30 generations. The crossover probability was set to 0.80 and the mutation probability to 0.20. A separate results folder was created so that the progress, selected descriptors and checkpoints could be saved during the long run.

In [ ]:
%%writefile full_ga_run.py

import time
import random
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

from deap import base, creator, tools
from xgboost import XGBClassifier
from sklearn.metrics import matthews_corrcoef
from sklearn.utils.class_weight import compute_sample_weight


# Set the main folder and create a folder for the GA results
main_folder = Path.cwd()
results_folder = main_folder / "full_ga_results"
results_folder.mkdir(exist_ok=True)

checkpoint_file = results_folder / "full_ga_checkpoint.pkl"


# Load the training and validation data
X_train = pd.read_pickle(
    main_folder / "mordred_training_features_filtered.pkl"
).astype(np.float32)

X_valid = pd.read_pickle(
    main_folder / "mordred_validation_features_filtered.pkl"
).astype(np.float32)

y_train = pd.read_pickle(
    main_folder / "mordred_training_labels.pkl"
).squeeze().astype(np.int8)

y_valid = pd.read_pickle(
    main_folder / "mordred_validation_labels.pkl"
).squeeze().astype(np.int8)

descriptor_names_df = pd.read_csv(
    main_folder / "mordred_filtered_descriptor_names.csv"
)

descriptor_names_df = descriptor_names_df.loc[
    :,
    ~descriptor_names_df.columns.str.startswith("Unnamed")
]

descriptor_names = (
    descriptor_names_df.iloc[:, 0]
    .astype(str)
    .tolist()
)


# Convert the datasets to NumPy arrays for repeated model training
X_train_array = X_train.to_numpy(copy=False)
X_valid_array = X_valid.to_numpy(copy=False)

y_train_array = y_train.to_numpy()
y_valid_array = y_valid.to_numpy()

train_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_train_array
)


# Use the same XGBoost settings as the mRMR assessment
xgb_params = {
    "objective": "binary:logistic",
    "n_estimators": 1000,
    "learning_rate": 0.05,
    "max_depth": 6,
    "min_child_weight": 3,
    "subsample": 0.80,
    "colsample_bytree": 0.80,
    "reg_alpha": 0.10,
    "reg_lambda": 1.00,
    "tree_method": "hist",
    "eval_metric": "logloss",
    "early_stopping_rounds": 50,
    "random_state": 42,
    "n_jobs": -1
}


# Full GA settings
population_size = 40
number_of_generations = 30
crossover_probability = 0.80
mutation_probability = 0.20
chromosome_length = 626


# Store evaluated feature subsets so they are not trained twice
fitness_cache = {}


def evaluate_feature_subset(individual):
    feature_mask = np.asarray(individual, dtype=bool)

    # A chromosome must contain at least one descriptor
    if feature_mask.sum() == 0:
        return -1.0

    cache_key = feature_mask.tobytes()

    if cache_key in fitness_cache:
        return fitness_cache[cache_key]

    X_train_selected = X_train_array[:, feature_mask]
    X_valid_selected = X_valid_array[:, feature_mask]

    model = XGBClassifier(**xgb_params)

    model.fit(
        X_train_selected,
        y_train_array,
        sample_weight=train_weights,
        eval_set=[(X_valid_selected, y_valid_array)],
        verbose=False
    )

    validation_probabilities = model.predict_proba(
        X_valid_selected
    )[:, 1]

    validation_predictions = (
        validation_probabilities >= 0.50
    ).astype(np.int8)

    mcc = matthews_corrcoef(
        y_valid_array,
        validation_predictions
    )

    fitness_cache[cache_key] = float(mcc)

    return float(mcc)


# Create the DEAP classes
if not hasattr(creator, "FullGAFitness"):
    creator.create(
        "FullGAFitness",
        base.Fitness,
        weights=(1.0,)
    )

if not hasattr(creator, "FullGAIndividual"):
    creator.create(
        "FullGAIndividual",
        list,
        fitness=creator.FullGAFitness
    )


toolbox = base.Toolbox()

toolbox.register(
    "attribute",
    random.randint,
    0,
    1
)

toolbox.register(
    "individual",
    tools.initRepeat,
    creator.FullGAIndividual,
    toolbox.attribute,
    n=chromosome_length
)

toolbox.register(
    "population",
    tools.initRepeat,
    list,
    toolbox.individual
)


def ga_fitness(individual):
    return (evaluate_feature_subset(individual),)


toolbox.register("evaluate", ga_fitness)
toolbox.register("mate", tools.cxTwoPoint)

toolbox.register(
    "mutate",
    tools.mutFlipBit,
    indpb=1 / chromosome_length
)

toolbox.register(
    "select",
    tools.selTournament,
    tournsize=3
)


def evaluate_invalid_individuals(population):
    invalid_individuals = [
        individual
        for individual in population
        if not individual.fitness.valid
    ]

    fitness_values = map(
        toolbox.evaluate,
        invalid_individuals
    )

    for individual, fitness_value in zip(
        invalid_individuals,
        fitness_values
    ):
        individual.fitness.values = fitness_value

    return len(invalid_individuals)


def save_progress(
    generation,
    population,
    hall_of_fame,
    history,
    evaluations,
    elapsed_seconds
):
    hall_of_fame.update(population)

    fitness_values = np.array([
        individual.fitness.values[0]
        for individual in population
    ])

    best_individual = hall_of_fame[0]

    history.append({
        "generation": generation,
        "evaluations": evaluations,
        "best_mcc": best_individual.fitness.values[0],
        "mean_mcc": fitness_values.mean(),
        "minimum_mcc": fitness_values.min(),
        "maximum_mcc": fitness_values.max(),
        "selected_descriptors": sum(best_individual),
        "unique_subsets_evaluated": len(fitness_cache),
        "elapsed_minutes": elapsed_seconds / 60
    })

    pd.DataFrame(history).to_csv(
        results_folder / "full_ga_history.csv",
        index=False
    )

    selected_descriptor_names = [
        descriptor_names[index]
        for index, selected in enumerate(best_individual)
        if selected == 1
    ]

    pd.DataFrame({
        "descriptor": selected_descriptor_names
    }).to_csv(
        results_folder / "full_ga_best_features.csv",
        index=False
    )

    np.save(
        results_folder / "full_ga_best_chromosome.npy",
        np.asarray(best_individual, dtype=np.int8)
    )

    checkpoint = {
        "generation": generation,
        "population": population,
        "hall_of_fame": hall_of_fame,
        "history": history,
        "fitness_cache": fitness_cache,
        "random_state": random.getstate(),
        "numpy_random_state": np.random.get_state(),
        "elapsed_seconds": elapsed_seconds
    }

    with open(checkpoint_file, "wb") as checkpoint_output:
        pickle.dump(checkpoint, checkpoint_output)

    print(
        f"Generation {generation} | "
        f"Evaluations: {evaluations} | "
        f"Best MCC: {best_individual.fitness.values[0]:.4f} | "
        f"Selected: {sum(best_individual)} | "
        f"Unique subsets: {len(fitness_cache)} | "
        f"Elapsed: {elapsed_seconds / 60:.1f} minutes",
        flush=True
    )


# Continue from the checkpoint if the run was interrupted
if checkpoint_file.exists():
    print("Loading the saved GA checkpoint.", flush=True)

    with open(checkpoint_file, "rb") as checkpoint_input:
        checkpoint = pickle.load(checkpoint_input)

    population = checkpoint["population"]
    hall_of_fame = checkpoint["hall_of_fame"]
    history = checkpoint["history"]
    fitness_cache = checkpoint["fitness_cache"]

    random.setstate(checkpoint["random_state"])
    np.random.set_state(checkpoint["numpy_random_state"])

    completed_generation = checkpoint["generation"]
    previous_elapsed_seconds = checkpoint["elapsed_seconds"]

    first_generation = completed_generation + 1

else:
    print("Starting a new full GA run.", flush=True)

    random.seed(42)
    np.random.seed(42)

    population = toolbox.population(n=population_size)
    hall_of_fame = tools.HallOfFame(1)
    history = []

    previous_elapsed_seconds = 0
    first_generation = 1

    initial_start = time.time()

    initial_evaluations = evaluate_invalid_individuals(
        population
    )

    initial_elapsed = (
        previous_elapsed_seconds
        + time.time()
        - initial_start
    )

    save_progress(
        generation=0,
        population=population,
        hall_of_fame=hall_of_fame,
        history=history,
        evaluations=initial_evaluations,
        elapsed_seconds=initial_elapsed
    )

    previous_elapsed_seconds = initial_elapsed


# Run the remaining generations
run_start_time = time.time()

for generation in range(
    first_generation,
    number_of_generations + 1
):
    elite = toolbox.clone(hall_of_fame[0])

    offspring = toolbox.select(
        population,
        len(population)
    )

    offspring = list(map(toolbox.clone, offspring))

    # Apply crossover
    for first, second in zip(
        offspring[::2],
        offspring[1::2]
    ):
        if random.random() < crossover_probability:
            toolbox.mate(first, second)

            if first.fitness.valid:
                del first.fitness.values

            if second.fitness.valid:
                del second.fitness.values

    # Apply mutation
    for individual in offspring:
        if random.random() < mutation_probability:
            toolbox.mutate(individual)

            if individual.fitness.valid:
                del individual.fitness.values

    generation_evaluations = evaluate_invalid_individuals(
        offspring
    )

    # Keep the best result found so far
    worst_index = min(
        range(len(offspring)),
        key=lambda index: offspring[index].fitness.values[0]
    )

    if (
        elite.fitness.values[0]
        > offspring[worst_index].fitness.values[0]
    ):
        offspring[worst_index] = elite

    population[:] = offspring

    total_elapsed_seconds = (
        previous_elapsed_seconds
        + time.time()
        - run_start_time
    )

    save_progress(
        generation=generation,
        population=population,
        hall_of_fame=hall_of_fame,
        history=history,
        evaluations=generation_evaluations,
        elapsed_seconds=total_elapsed_seconds
    )


best_individual = hall_of_fame[0]

print("\nFull GA completed.", flush=True)
print(
    "Best validation MCC:",
    round(best_individual.fitness.values[0], 4),
    flush=True
)
print(
    "Number of selected descriptors:",
    sum(best_individual),
    flush=True
)
print(
    "Unique subsets evaluated:",
    len(fitness_cache),
    flush=True
)

## Full GA script

The full Genetic Algorithm code was saved as `full_ga_run.py`. The script includes the fixed XGBoost settings, checkpoint saving and automatic continuation if the run is interrupted.

In [ ]:
from pathlib import Path
import subprocess
import sys

# Check that the script exists and has no syntax errors
script_file = Path("full_ga_run.py")

print("Script found:", script_file.exists())

syntax_check = subprocess.run(
    [sys.executable, "-m", "py_compile", str(script_file)],
    capture_output=True,
    text=True
)

if syntax_check.returncode == 0:
    print("Syntax check passed.")
else:
    print("Syntax error:")
    print(syntax_check.stderr)

## Saved GA outputs

All expected GA result files were saved successfully. The checkpoint, generation history, final chromosome, selected descriptor names and run log are available. The old PID file is harmless because the completed log confirms that the GA finished normally.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

results_dir = Path("full_ga_results")

history = pd.read_csv(results_dir / "full_ga_history.csv")
best_features = pd.read_csv(results_dir / "full_ga_best_features.csv")
best_chromosome = np.load(
    results_dir / "full_ga_best_chromosome.npy"
)

print("History shape:", history.shape)
print("Best features shape:", best_features.shape)
print("Chromosome length:", len(best_chromosome))
print("Selected descriptors:", int(best_chromosome.sum()))

print("\nHistory columns:")
print(history.columns.tolist())

print("\nLast five generations:")
display(history.tail())

print("\nFirst ten selected descriptors:")
display(best_features.head(10))

## GA result verification

The saved GA outputs are internally consistent:

- The history contains 31 rows, representing Generation 0 through Generation 30.
- The chromosome contains 626 positions, matching the full filtered Mordred feature pool.
- Exactly 316 positions are selected, matching the 316 descriptor names saved in the feature file.
- The final best validation MCC remained 0.712946.
- The GA evaluated 469 unique feature subsets.

This confirms that the final GA-selected feature set was saved correctly. The test set has still not been used.

In [ ]:
# Load the saved split and previous test predictions
split_data = pd.read_csv("split_assignments.csv")
test_predictions = pd.read_csv(
    "final_mordred_mrmr_test_predictions.csv"
)

# Exact test rows from the original scaffold split
test_indices = split_data.index[
    split_data["split"] == "test"
].to_numpy()

predicted_test_indices = test_predictions[
    "row_index"
].to_numpy()

print("Test rows in split file:", len(test_indices))
print("Test rows in prediction file:", len(predicted_test_indices))

print(
    "Indices match in the same order:",
    np.array_equal(test_indices, predicted_test_indices)
)

print(
    "Indices contain the same rows:",
    set(test_indices) == set(predicted_test_indices)
)

# Verify canonical SMILES
split_test_smiles = split_data.loc[
    test_indices, "canonical_smiles"
].to_numpy()

prediction_smiles = test_predictions[
    "canonical_smiles"
].to_numpy()

print(
    "Canonical SMILES match:",
    np.array_equal(split_test_smiles, prediction_smiles)
)

# Inspect the saved Mordred preprocessing files
training_clean = pd.read_pickle(
    "mordred_training_features_clean.pkl"
)

training_filtered = pd.read_pickle(
    "mordred_training_features_filtered.pkl"
)

validation_filtered = pd.read_pickle(
    "mordred_validation_features_filtered.pkl"
)

print("\nTraining clean shape:", training_clean.shape)
print("Training filtered shape:", training_filtered.shape)
print("Validation filtered shape:", validation_filtered.shape)

print(
    "Training and validation columns match:",
    training_filtered.columns.equals(
        validation_filtered.columns
    )
)

print("\nTraining clean index sample:")
print(training_clean.index[:10].tolist())

print("\nTraining filtered index sample:")
print(training_filtered.index[:10].tolist())

# Find possible preprocessing files
print("\nPossible preprocessing files:")

keywords = [
    "median",
    "imput",
    "variance",
    "correlation",
    "filtered_descriptor",
    "preprocess"
]

for path in sorted(Path(".").rglob("*")):
    if path.is_file():
        filename = path.name.lower()

        if any(word in filename for word in keywords):
            print(path)

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Load the original split and previous test labels
split_data = pd.read_csv("split_assignments.csv")
test_predictions = pd.read_csv(
    "final_mordred_mrmr_test_predictions.csv"
)

# Exact untouched test indices
test_indices = split_data.index[
    split_data["split"] == "test"
].to_numpy()

# Load preprocessing information learned from training data
training_clean = pd.read_pickle(
    "mordred_training_features_clean.pkl"
)

clean_columns = training_clean.columns.tolist()

training_medians = pd.read_csv(
    "mordred_training_medians.csv",
    index_col=0
)["training_median"]

training_medians = training_medians.reindex(clean_columns)

filtered_columns = pd.read_csv(
    "mordred_filtered_descriptor_names.csv"
)["descriptor"].tolist()

print("Test rows required:", len(test_indices))
print("Clean descriptors required:", len(clean_columns))
print("Filtered descriptors required:", len(filtered_columns))
print("Missing saved medians:", int(training_medians.isna().sum()))

# Read only test rows from each Mordred chunk
test_parts = []

chunk_files = sorted(
    Path("mordred_numeric_chunks").glob(
        "mordred_numeric_*.pkl"
    )
)

for chunk_file in chunk_files:
    chunk = pd.read_pickle(chunk_file)

    required_rows = chunk.index.intersection(test_indices)

    if len(required_rows) > 0:
        test_parts.append(
            chunk.loc[required_rows, clean_columns]
        )

# Combine and restore the exact original test order
X_test_clean = pd.concat(test_parts, axis=0)
X_test_clean = X_test_clean.loc[test_indices]

# Apply the same preprocessing used for training
X_test_clean = X_test_clean.replace(
    [np.inf, -np.inf],
    np.nan
)

X_test_clean = X_test_clean.fillna(training_medians)

# Keep the final 626 filtered descriptors
X_test = X_test_clean[
    filtered_columns
].astype("float32")

# Recover the exact test labels in the same row order
y_test = (
    test_predictions
    .set_index("row_index")
    .loc[test_indices, "true_label"]
    .astype("int8")
)

# Final checks
print("\nTest feature shape:", X_test.shape)
print("Test label shape:", y_test.shape)
print("Duplicate test indices:", X_test.index.duplicated().sum())
print("Missing values:", int(X_test.isna().sum().sum()))
print(
    "Infinite values:",
    int(np.isinf(X_test.to_numpy()).sum())
)
print(
    "Feature columns match training:",
    X_test.columns.equals(
        pd.read_pickle(
            "mordred_training_features_filtered.pkl"
        ).columns
    )
)

# Save for final model evaluation
X_test.to_pickle(
    "mordred_test_features_filtered.pkl"
)

y_test.to_pickle(
    "mordred_test_labels.pkl"
)

print("\nSaved:")
print("mordred_test_features_filtered.pkl")
print("mordred_test_labels.pkl")

## Untouched test set prepared

The exact 14,700-row scaffold test set was reconstructed successfully using the original split assignments and preprocessing values learned only from the training data. It contains the same 626 filtered Mordred descriptors as the training and validation sets, with no missing or infinite values.

The test set has now been saved, but it has not yet been used to evaluate the GA model.

In [ ]:
import pandas as pd

# Load training, validation and untouched test data
X_train = pd.read_pickle(
    "mordred_training_features_filtered.pkl"
).astype("float32")

y_train = pd.read_pickle(
    "mordred_training_labels.pkl"
).squeeze().astype("int8")

X_valid = pd.read_pickle(
    "mordred_validation_features_filtered.pkl"
).astype("float32")

y_valid = pd.read_pickle(
    "mordred_validation_labels.pkl"
).squeeze().astype("int8")

X_test = pd.read_pickle(
    "mordred_test_features_filtered.pkl"
).astype("float32")

y_test = pd.read_pickle(
    "mordred_test_labels.pkl"
).squeeze().astype("int8")

# Load the 316 descriptors selected by GA
ga_features = pd.read_csv(
    "full_ga_results/full_ga_best_features.csv"
)["descriptor"].tolist()

# Combine training and validation for the final model
X_final_train = pd.concat(
    [X_train[ga_features], X_valid[ga_features]],
    axis=0
)

y_final_train = pd.concat(
    [y_train, y_valid],
    axis=0
)

# Prepare untouched test features
X_test_ga = X_test[ga_features]

print("GA descriptors:", len(ga_features))
print("Final training shape:", X_final_train.shape)
print("Final training labels:", y_final_train.shape)
print("GA test shape:", X_test_ga.shape)
print("Test labels:", y_test.shape)

print("\nFinal training label counts:")
print(y_final_train.value_counts().sort_index())

print("\nTest label counts:")
print(y_test.value_counts().sort_index())

print(
    "\nTraining and test feature order matches:",
    X_final_train.columns.equals(X_test_ga.columns)
)

## Final GA model data

The final datasets are correctly aligned:

- Training + validation: 82,071 molecules
- Untouched scaffold test: 14,700 molecules
- GA-selected descriptors: 316
- Feature names and order match between training and test data

The final XGBoost model will now be trained using the same fixed hyperparameters used during GA. A threshold of **0.50** will be used because the GA fitness was validation MCC calculated at 0.50.

In [ ]:
import pandas as pd
import numpy as np

from xgboost import XGBClassifier
from sklearn.metrics import (
    balanced_accuracy_score,
    matthews_corrcoef,
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Final model with the same fixed settings used during GA
ga_final_model = XGBClassifier(
    objective="binary:logistic",
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=3,
    subsample=0.80,
    colsample_bytree=0.80,
    reg_alpha=0.10,
    reg_lambda=1.00,
    tree_method="hist",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

# Train using training and validation data together
ga_final_model.fit(
    X_final_train,
    y_final_train
)

# Predict on the untouched test set
ga_test_probability = ga_final_model.predict_proba(
    X_test_ga
)[:, 1]

ga_threshold = 0.50

ga_test_prediction = (
    ga_test_probability >= ga_threshold
).astype(int)

# Confusion matrix
tn, fp, fn, tp = confusion_matrix(
    y_test,
    ga_test_prediction,
    labels=[0, 1]
).ravel()

specificity = tn / (tn + fp)

# Test results
ga_test_results = pd.DataFrame([{
    "descriptor_method": "Mordred",
    "feature_selection": "Genetic Algorithm",
    "model": "XGBoost",
    "number_of_features": len(ga_features),
    "threshold": ga_threshold,
    "training_rows": len(X_final_train),
    "test_rows": len(X_test_ga),
    "balanced_accuracy": balanced_accuracy_score(
        y_test, ga_test_prediction
    ),
    "mcc": matthews_corrcoef(
        y_test, ga_test_prediction
    ),
    "roc_auc": roc_auc_score(
        y_test, ga_test_probability
    ),
    "average_precision": average_precision_score(
        y_test, ga_test_probability
    ),
    "precision": precision_score(
        y_test, ga_test_prediction,
        zero_division=0
    ),
    "recall": recall_score(
        y_test, ga_test_prediction,
        zero_division=0
    ),
    "specificity": specificity,
    "f1_score": f1_score(
        y_test, ga_test_prediction,
        zero_division=0
    ),
    "true_negative": tn,
    "false_positive": fp,
    "false_negative": fn,
    "true_positive": tp
}])

# Add row information to the prediction file
test_metadata = pd.read_csv(
    "final_mordred_mrmr_test_predictions.csv"
).set_index("row_index").loc[X_test_ga.index]

ga_test_predictions = pd.DataFrame({
    "row_index": X_test_ga.index,
    "canonical_smiles": test_metadata[
        "canonical_smiles"
    ].to_numpy(),
    "true_label": y_test.to_numpy(),
    "predicted_probability": ga_test_probability,
    "predicted_label": ga_test_prediction,
    "threshold": ga_threshold
})

# Save results, predictions and model
ga_test_results.to_csv(
    "final_mordred_ga_test_results.csv",
    index=False
)

ga_test_predictions.to_csv(
    "final_mordred_ga_test_predictions.csv",
    index=False
)

ga_final_model.save_model(
    "final_mordred_ga_xgboost_model.json"
)

print("Final GA test results:")
display(ga_test_results)

print("\nSaved:")
print("final_mordred_ga_test_results.csv")
print("final_mordred_ga_test_predictions.csv")
print("final_mordred_ga_xgboost_model.json")

In [ ]:
# Reload the mRMR descriptor file
mrmr_file = pd.read_csv(
    "final_mordred_mrmr_300_descriptors.csv"
)

# Remove any saved index column
mrmr_file = mrmr_file.loc[
    :,
    ~mrmr_file.columns.str.startswith("Unnamed")
]

print("Columns:", mrmr_file.columns.tolist())
display(mrmr_file.head())

# Check which column contains descriptor names
column_matches = {}

for column in mrmr_file.columns:
    values = mrmr_file[column].astype(str)
    column_matches[column] = values.isin(
        X_final_all.columns
    ).sum()

print("\nDescriptor matches by column:")
for column, matches in column_matches.items():
    print(column, ":", matches)

# Select the column with the most descriptor matches
descriptor_column = max(
    column_matches,
    key=column_matches.get
)

mrmr_features = (
    mrmr_file[descriptor_column]
    .astype(str)
    .tolist()
)

print("\nSelected descriptor column:", descriptor_column)
print("Number of mRMR descriptors:", len(mrmr_features))
print(
    "All mRMR descriptors available:",
    all(
        descriptor in X_final_all.columns
        for descriptor in mrmr_features
    )
)

In [ ]:
import pandas as pd
import numpy as np

from xgboost import XGBClassifier
from sklearn.metrics import (
    balanced_accuracy_score,
    matthews_corrcoef,
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    confusion_matrix,
    f1_score
)

# Same XGBoost settings for all three models
fair_xgb_params = {
    "objective": "binary:logistic",
    "n_estimators": 1000,
    "learning_rate": 0.05,
    "max_depth": 6,
    "min_child_weight": 3,
    "subsample": 0.80,
    "colsample_bytree": 0.80,
    "reg_alpha": 0.10,
    "reg_lambda": 1.00,
    "tree_method": "hist",
    "eval_metric": "logloss",
    "random_state": 42,
    "n_jobs": -1
}

threshold = 0.50

# Feature sets to compare
feature_sets = {
    "Full Mordred": X_final_all.columns.tolist(),
    "mRMR": mrmr_features,
    "Genetic Algorithm": ga_features
}

comparison_results = []
comparison_predictions = {}

for method, features in feature_sets.items():

    print(
        f"\nTraining {method} model "
        f"with {len(features)} descriptors..."
    )

    model = XGBClassifier(**fair_xgb_params)

    # Train using the same rows and balanced weights
    model.fit(
        X_final_all[features],
        y_final,
        sample_weight=final_weights
    )

    # Predict on the same untouched scaffold test set
    probabilities = model.predict_proba(
        X_test[features]
    )[:, 1]

    predictions = (
        probabilities >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        predictions,
        labels=[0, 1]
    ).ravel()

    specificity = tn / (tn + fp)

    comparison_results.append({
        "feature_selection": method,
        "number_of_features": len(features),
        "threshold": threshold,
        "training_rows": len(X_final_all),
        "test_rows": len(X_test),
        "balanced_accuracy": balanced_accuracy_score(
            y_test,
            predictions
        ),
        "mcc": matthews_corrcoef(
            y_test,
            predictions
        ),
        "roc_auc": roc_auc_score(
            y_test,
            probabilities
        ),
        "average_precision": average_precision_score(
            y_test,
            probabilities
        ),
        "precision": precision_score(
            y_test,
            predictions,
            zero_division=0
        ),
        "recall": recall_score(
            y_test,
            predictions,
            zero_division=0
        ),
        "specificity": specificity,
        "f1_score": f1_score(
            y_test,
            predictions,
            zero_division=0
        ),
        "true_negative": tn,
        "false_positive": fp,
        "false_negative": fn,
        "true_positive": tp
    })

    comparison_predictions[method] = {
        "probabilities": probabilities,
        "predictions": predictions
    }

    safe_name = (
        method.lower()
        .replace(" ", "_")
    )

    model.save_model(
        f"fair_{safe_name}_xgboost_model.json"
    )

# Create the final comparison table
fair_comparison = pd.DataFrame(
    comparison_results
).sort_values(
    "mcc",
    ascending=False
).reset_index(drop=True)

fair_comparison.to_csv(
    "fair_full_mrmr_ga_test_comparison.csv",
    index=False
)

print("\nFair test comparison:")
display(fair_comparison)

print("\nSaved:")
print("fair_full_mrmr_ga_test_comparison.csv")
print("Three separately trained XGBoost model files")

In [ ]:
import pandas as pd

# Load the fair comparison
final_comparison = pd.read_csv(
    "fair_full_mrmr_ga_test_comparison.csv"
)

best_mcc = final_comparison["mcc"].max()

# Add useful comparison columns
final_comparison["feature_reduction_vs_626_percent"] = (
    1 - final_comparison["number_of_features"] / 626
) * 100

final_comparison["mcc_difference_from_best"] = (
    best_mcc - final_comparison["mcc"]
)

final_comparison["mcc_rank"] = (
    final_comparison["mcc"]
    .rank(method="min", ascending=False)
    .astype(int)
)

# Arrange the most useful columns first
first_columns = [
    "mcc_rank",
    "feature_selection",
    "number_of_features",
    "feature_reduction_vs_626_percent",
    "threshold",
    "balanced_accuracy",
    "mcc",
    "mcc_difference_from_best",
    "roc_auc",
    "average_precision",
    "precision",
    "recall",
    "specificity",
    "f1_score"
]

remaining_columns = [
    column for column in final_comparison.columns
    if column not in first_columns
]

final_comparison = final_comparison[
    first_columns + remaining_columns
].sort_values("mcc_rank")

# Correct GA validation-to-test comparison
ga_history = pd.read_csv(
    "full_ga_results/full_ga_history.csv"
)

ga_validation_mcc = ga_history["best_mcc"].max()

ga_fair_test_mcc = final_comparison.loc[
    final_comparison["feature_selection"]
    == "Genetic Algorithm",
    "mcc"
].iloc[0]

ga_mcc_drop = ga_validation_mcc - ga_fair_test_mcc

corrected_overfitting_check = pd.DataFrame([{
    "model": "Mordred GA XGBoost",
    "number_of_features": 316,
    "validation_mcc": ga_validation_mcc,
    "fair_test_mcc": ga_fair_test_mcc,
    "validation_to_test_mcc_drop": ga_mcc_drop,
    "assessment": (
        "Small generalisation drop; "
        "no evidence of severe overfitting"
    )
}])

# Save the corrected files
final_comparison.to_csv(
    "final_fair_feature_selection_comparison.csv",
    index=False
)

corrected_overfitting_check.to_csv(
    "ga_corrected_overfitting_assessment.csv",
    index=False
)

print("Final fair comparison:")
display(final_comparison)

print("\nCorrected GA overfitting assessment:")
display(corrected_overfitting_check)

print("\nSaved:")
print("final_fair_feature_selection_comparison.csv")
print("ga_corrected_overfitting_assessment.csv")

## Final fair comparison confirmed

The corrected comparison is now complete and consistent.

- The full 626-descriptor model achieved the highest test MCC: **0.7012**.
- mRMR reduced the features to 300, a **52.1% reduction**, with only a **0.0026 decrease in MCC**. It also achieved the highest balanced accuracy.
- GA selected 316 descriptors, a **49.5% reduction**, with a test MCC of **0.6974**.
- The GA validation-to-test MCC decrease was **0.0155**, indicating a small generalisation drop but no evidence of severe overfitting.

Therefore, GA successfully removed about half of the descriptors while preserving most predictive performance, but mRMR produced a slightly smaller and slightly better-performing subset.

The next comparison is to examine how many descriptors were selected by both mRMR and GA, as requested by the supervisor.

In [ ]:
import pandas as pd

# Load the mRMR-selected descriptors
mrmr_data = pd.read_csv(
    "final_mordred_mrmr_300_descriptors.csv"
)

mrmr_features = (
    mrmr_data["descriptor"]
    .astype(str)
    .tolist()
)

# Load the GA-selected descriptors
ga_features = pd.read_csv(
    "full_ga_results/full_ga_best_features.csv"
)["descriptor"].astype(str).tolist()

mrmr_set = set(mrmr_features)
ga_set = set(ga_features)

# Compare the two feature-selection methods
shared_features = sorted(mrmr_set & ga_set)
mrmr_only = sorted(mrmr_set - ga_set)
ga_only = sorted(ga_set - mrmr_set)
all_selected = mrmr_set | ga_set

jaccard_similarity = (
    len(shared_features) / len(all_selected)
)

overlap_summary = pd.DataFrame([{
    "mrmr_features": len(mrmr_set),
    "ga_features": len(ga_set),
    "shared_features": len(shared_features),
    "mrmr_only_features": len(mrmr_only),
    "ga_only_features": len(ga_only),
    "jaccard_similarity": jaccard_similarity,
    "mrmr_features_shared_percent": (
        len(shared_features) / len(mrmr_set)
    ) * 100,
    "ga_features_shared_percent": (
        len(shared_features) / len(ga_set)
    ) * 100
}])

# Save the descriptor comparisons
pd.DataFrame({
    "descriptor": shared_features
}).to_csv(
    "mrmr_ga_shared_descriptors.csv",
    index=False
)

pd.DataFrame({
    "descriptor": mrmr_only
}).to_csv(
    "mrmr_only_descriptors.csv",
    index=False
)

pd.DataFrame({
    "descriptor": ga_only
}).to_csv(
    "ga_only_descriptors.csv",
    index=False
)

overlap_summary.to_csv(
    "mrmr_ga_descriptor_overlap_summary.csv",
    index=False
)

print("mRMR and GA descriptor overlap:")
display(overlap_summary)

print("\nFirst ten shared descriptors:")
display(
    pd.DataFrame({
        "descriptor": shared_features[:10]
    })
)

print("\nSaved:")
print("mrmr_ga_descriptor_overlap_summary.csv")
print("mrmr_ga_shared_descriptors.csv")
print("mrmr_only_descriptors.csv")
print("ga_only_descriptors.csv")

## mRMR and GA descriptor overlap

The two methods selected substantially different feature subsets:

- mRMR selected 300 descriptors.
- GA selected 316 descriptors.
- Only 145 descriptors were selected by both methods.
- 155 descriptors were unique to mRMR.
- 171 descriptors were unique to GA.
- The Jaccard similarity was 0.308.

Therefore, only about half of each method’s selected descriptors overlapped. Despite this relatively limited overlap, both models produced very similar test performance. This suggests that the Mordred dataset contains correlated or partly interchangeable descriptors, allowing different feature-selection methods to identify different but similarly predictive subsets.

For the dissertation, state that the GA did not simply reproduce the mRMR subset; it independently found an alternative set of descriptors from the original 626-feature pool.

In [ ]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    matthews_corrcoef,
    balanced_accuracy_score
)

# Balanced weights for the original training set
train_weights_check = compute_sample_weight(
    class_weight="balanced",
    y=y_train
)

# Same fixed model settings
overfit_xgb_params = {
    "objective": "binary:logistic",
    "n_estimators": 1000,
    "learning_rate": 0.05,
    "max_depth": 6,
    "min_child_weight": 3,
    "subsample": 0.80,
    "colsample_bytree": 0.80,
    "reg_alpha": 0.10,
    "reg_lambda": 1.00,
    "tree_method": "hist",
    "eval_metric": "logloss",
    "early_stopping_rounds": 50,
    "random_state": 42,
    "n_jobs": -1
}

feature_sets = {
    "Full Mordred": X_train.columns.tolist(),
    "mRMR": mrmr_features,
    "Genetic Algorithm": ga_features
}

overfitting_results = []

for method, features in feature_sets.items():

    print(f"Checking {method}...")

    model = XGBClassifier(**overfit_xgb_params)

    model.fit(
        X_train[features],
        y_train,
        sample_weight=train_weights_check,
        eval_set=[(X_valid[features], y_valid)],
        verbose=False
    )

    train_probabilities = model.predict_proba(
        X_train[features]
    )[:, 1]

    valid_probabilities = model.predict_proba(
        X_valid[features]
    )[:, 1]

    train_predictions = (
        train_probabilities >= 0.50
    ).astype(int)

    valid_predictions = (
        valid_probabilities >= 0.50
    ).astype(int)

    train_mcc = matthews_corrcoef(
        y_train,
        train_predictions
    )

    validation_mcc = matthews_corrcoef(
        y_valid,
        valid_predictions
    )

    overfitting_results.append({
        "feature_selection": method,
        "number_of_features": len(features),
        "best_iteration": model.best_iteration,
        "training_mcc": train_mcc,
        "validation_mcc": validation_mcc,
        "training_validation_mcc_gap": (
            train_mcc - validation_mcc
        ),
        "training_balanced_accuracy":
            balanced_accuracy_score(
                y_train,
                train_predictions
            ),
        "validation_balanced_accuracy":
            balanced_accuracy_score(
                y_valid,
                valid_predictions
            )
    })

overfitting_comparison = pd.DataFrame(
    overfitting_results
)

overfitting_comparison.to_csv(
    "full_mrmr_ga_overfitting_diagnostic.csv",
    index=False
)

display(overfitting_comparison)

print(
    "\nSaved: "
    "full_mrmr_ga_overfitting_diagnostic.csv"
)